# Ch.2 — Logistic Regression

> Take the linear score from Ch.1, squash it through a sigmoid, and you have a probability. Everything else — loss, gradient, optimiser — follows the same pattern.

**Dataset:** California Housing — `sklearn.datasets.fetch_california_housing`  
**Task:** Binary classify — is a district **high-value** (above median house price)?

## The Core Idea

Logistic regression adds one step to the Ch.1 pipeline:

$$z = \mathbf{W}^\top \mathbf{x} + b \qquad \hat{p} = \sigma(z) = \frac{1}{1+e^{-z}} \qquad \hat{y} = \mathbf{1}[\hat{p} \geq \tau]$$

- $z$ — **logit**: unbounded linear score  
- $\hat{p}$ — **probability**: always in $(0, 1)$  
- $\tau$ — **decision threshold**: the dial you tune to trade precision vs. recall

Training loss: **Binary Cross-Entropy**  
$$\mathcal{L} = -\frac{1}{n}\sum\left[y_i\log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\right]$$

## Running Example

The real estate platform needs a binary signal: is a California district **high-value**?  
We define high-value as `MedHouseVal > median(MedHouseVal)` — exactly 50 % of districts qualify, giving a balanced binary problem.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
print("Libraries loaded.")

In [ ]:
# ── Load and build binary target ───────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df      = housing.frame

median_val = df["MedHouseVal"].median()
df["high_value"] = (df["MedHouseVal"] > median_val).astype(int)

print(f"Median house value threshold: ${median_val * 100_000:,.0f}")
print(f"Class distribution:\n{df['high_value'].value_counts()}")

X = df[housing.feature_names].values
y = df["high_value"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"\nTrain: {X_train.shape}  Test: {X_test.shape}")

## The Math — Sigmoid and BCE from Scratch

In [ ]:
# ── Sigmoid and BCE implementations ───────────────────────────────────────
def sigmoid(z):
    """Clip for numerical stability at extreme z values."""
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def bce_loss(y_true, p_hat, eps=1e-15):
    p = np.clip(p_hat, eps, 1 - eps)   # avoid log(0)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

# Plot sigmoid shape
z_range = np.linspace(-8, 8, 300)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z_range, sigmoid(z_range), color="#2980b9", linewidth=2.5)
ax.axhline(0.5, color="#e74c3c", linestyle="--", label="σ(0) = 0.5 (decision boundary)")
ax.axvline(0, color="#95a5a6", linestyle=":")
ax.fill_between(z_range, sigmoid(z_range), 0.5, where=(z_range >= 0),
                alpha=0.1, color="#27ae60", label="Predict 1 (at τ=0.5)")
ax.fill_between(z_range, sigmoid(z_range), 0.5, where=(z_range < 0),
                alpha=0.1, color="#e74c3c", label="Predict 0 (at τ=0.5)")
ax.set(title="Sigmoid Function — σ(z) = 1 / (1 + e⁻ᶻ)",
       xlabel="z (logit)", ylabel="σ(z) — probability")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Why BCE instead of MSE? Visualise gradient magnitudes ─────────────────
p_range = np.linspace(0.01, 0.99, 200)

# Gradients when true label y = 1
mse_grad = 2 * (p_range - 1) * p_range * (1 - p_range)   # ∂MSE/∂z for y=1
bce_grad = p_range - 1                                     # ∂BCE/∂z for y=1  (= p̂ - y)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p_range, np.abs(mse_grad), label="|MSE gradient|  (small and vanishing near 0)",
        color="#e74c3c", linewidth=2)
ax.plot(p_range, np.abs(bce_grad), label="|BCE gradient|  (large when wrong)",
        color="#27ae60", linewidth=2)
ax.set(title="Gradient Magnitude vs Predicted Probability (y=1)",
       xlabel="p̂ (predicted probability)", ylabel="|Gradient|")
ax.legend()
plt.tight_layout()
plt.show()
print("When p̂ ≈ 0 but y=1 (very wrong prediction):")
print(f"  |MSE gradient| ≈ {abs(mse_grad[0]):.4f}")
print(f"  |BCE gradient| ≈ {abs(bce_grad[0]):.4f}  ← much larger, faster learning")

## Step by Step — Manual Logistic Regression via Gradient Descent

In [ ]:
# ── Logistic regression via gradient descent from scratch ──────────────────
def logistic_gd(X, y, alpha=0.1, epochs=300):
    n, p = X.shape
    W = np.zeros(p)
    b = 0.0
    history = []

    for epoch in range(epochs):
        z     = X @ W + b
        p_hat = sigmoid(z)
        error = p_hat - y              # gradient wrt z: p̂ - y

        dW = (1 / n) * X.T @ error
        db = (1 / n) * error.sum()

        W -= alpha * dW
        b -= alpha * db

        history.append(bce_loss(y, p_hat))

    return W, b, history

W_gd, b_gd, loss_hist = logistic_gd(X_train, y_train, alpha=0.1, epochs=300)

p_hat_gd  = sigmoid(X_test @ W_gd + b_gd)
y_pred_gd = (p_hat_gd >= 0.5).astype(int)
print(f"Manual GD  → Test AUC: {roc_auc_score(y_test, p_hat_gd):.4f}  "
      f"F1: {f1_score(y_test, y_pred_gd):.4f}")

In [ ]:
# ── Plot loss curve ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_hist, color="#2980b9", linewidth=2)
ax.axhline(loss_hist[-1], color="#e74c3c", linestyle="--",
           label=f"Final BCE: {loss_hist[-1]:.4f}")
ax.set(title="Training Loss Curve (Binary Cross-Entropy)",
       xlabel="Epoch", ylabel="BCE Loss")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── sklearn LogisticRegression for comparison ──────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred_sk    = model.predict(X_test)

print("sklearn LogisticRegression (τ = 0.5):")
print(classification_report(y_test, y_pred_sk,
                             target_names=["low-value", "high-value"]))

## Evaluation — Confusion Matrix, ROC, PR Curve

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_sk)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: low", "Pred: high"],
            yticklabels=["Actual: low", "Actual: high"],
            ax=ax)
ax.set(title="Confusion Matrix (τ = 0.5)")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"Precision: {tp/(tp+fp):.3f}   Recall: {tp/(tp+fn):.3f}")

In [ ]:
# ── ROC curve and PR curve side by side ───────────────────────────────────
fpr, tpr, roc_thresholds = roc_curve(y_test, y_pred_proba)
auc_roc = roc_auc_score(y_test, y_pred_proba)

precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)
auc_pr = average_precision_score(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, color="#2980b9", linewidth=2,
             label=f"AUC-ROC = {auc_roc:.4f}")
axes[0].plot([0, 1], [0, 1], color="#95a5a6", linestyle="--", label="Random (AUC=0.5)")
axes[0].set(title="ROC Curve", xlabel="False Positive Rate", ylabel="True Positive Rate")
axes[0].legend()

# PR
axes[1].plot(recall_curve, precision_curve, color="#27ae60", linewidth=2,
             label=f"AUC-PR = {auc_pr:.4f}")
axes[1].axhline(0.5, color="#95a5a6", linestyle="--", label="Random classifier")
axes[1].set(title="Precision–Recall Curve", xlabel="Recall", ylabel="Precision")
axes[1].legend()

plt.suptitle("High-Value District Classifier — Model Evaluation", y=1.02)
plt.tight_layout()
plt.show()

## The Hyperparameter Dial — Decision Threshold τ

Lowering τ increases recall (catch more high-value districts) at the cost of precision (more false alarms). The platform team gets to choose: do they want fewer false badges (↑ precision) or fewer missed high-value listings (↑ recall)?

In [ ]:
# ── Threshold sweep: precision, recall, F1 vs τ ────────────────────────────
thresholds = np.linspace(0.1, 0.9, 80)
precisions, recalls, f1s = [], [], []

for tau in thresholds:
    preds = (y_pred_proba >= tau).astype(int)
    precisions.append(precision_score(y_test, preds, zero_division=0))
    recalls.append(recall_score(y_test, preds, zero_division=0))
    f1s.append(f1_score(y_test, preds, zero_division=0))

best_tau_idx = np.argmax(f1s)
best_tau     = thresholds[best_tau_idx]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, precisions, color="#2980b9",  linewidth=2, label="Precision")
ax.plot(thresholds, recalls,   color="#27ae60",  linewidth=2, label="Recall")
ax.plot(thresholds, f1s,       color="#e74c3c",  linewidth=2, label="F1")
ax.axvline(best_tau, color="#f39c12", linestyle="--",
           label=f"Best F1 threshold: τ = {best_tau:.2f}")
ax.axvline(0.5, color="#95a5a6", linestyle=":", label="Default τ = 0.5")
ax.set(title="Precision, Recall, F1 vs Decision Threshold",
       xlabel="Threshold τ", ylabel="Score")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Best F1 threshold: {best_tau:.2f}")
print(f"  Precision: {precisions[best_tau_idx]:.3f}")
print(f"  Recall:    {recalls[best_tau_idx]:.3f}")
print(f"  F1:        {f1s[best_tau_idx]:.3f}")

## What Can Go Wrong — Trap Demo

### Trap: High accuracy on imbalanced data says nothing

The housing dataset is balanced (50/50), so accuracy is meaningful here. Let's simulate what happens with a heavily skewed dataset — the kind you will encounter with fraud detection, churn prediction, or rare event classification.

In [ ]:
# ── Simulate class imbalance: 90% low-value, 10% high-value ───────────────
rng = np.random.default_rng(SEED)

# Keep all positives, downsample negatives to create 90/10 split
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
keep_neg = rng.choice(neg_idx, size=len(pos_idx) * 9, replace=False)
imb_idx  = np.concatenate([pos_idx, keep_neg])

X_imb, y_imb = X[imb_idx], y[imb_idx]
# Flip: now label 1 is the minority (10%)
y_imb = 1 - y_imb   # now 10% are "high-value"

Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=SEED, stratify=y_imb
)
sc_i = StandardScaler()
Xtr_i = sc_i.fit_transform(Xtr_i)
Xte_i = sc_i.transform(Xte_i)

# A dumb model: always predict majority class (0)
y_dumb = np.zeros(len(yte_i), dtype=int)

# A real model
m_imb = LogisticRegression(max_iter=1000, random_state=SEED)
m_imb.fit(Xtr_i, ytr_i)
y_imb_pred = m_imb.predict(Xte_i)
y_imb_prob = m_imb.predict_proba(Xte_i)[:, 1]

from sklearn.metrics import accuracy_score
print("                  Accuracy    F1     AUC-ROC")
print("-" * 50)
print(f"Dumb (all zeros)    {accuracy_score(yte_i, y_dumb):.3f}     "
      f"{f1_score(yte_i, y_dumb, zero_division=0):.3f}     N/A")
print(f"Logistic Regr.      {accuracy_score(yte_i, y_imb_pred):.3f}     "
      f"{f1_score(yte_i, y_imb_pred):.3f}     "
      f"{roc_auc_score(yte_i, y_imb_prob):.3f}")
print("\n→ Dumb model achieves ~90% accuracy. Always report F1 + AUC on imbalanced data.")

In [ ]:
# ── Feature importance via coefficients ───────────────────────────────────
# Logistic regression coefficients are interpretable as log-odds contributions
coef_df = pd.DataFrame({
    "feature":     housing.feature_names,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors  = ["#27ae60" if c > 0 else "#e74c3c" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Logistic Regression Coefficients\n(log-odds contribution, features standardised)",
       xlabel="Coefficient value")
plt.tight_layout()
plt.show()
print("Positive coefficient → feature pushes toward high-value prediction")
print("Negative coefficient → feature pushes toward low-value prediction")

## Exercises

**Exercise 1 — Threshold for the business rule**  
The platform decides: the "premium neighbourhood" badge must have precision ≥ 0.90 (can't put a badge on a non-premium district). Find the threshold τ that achieves this and report the resulting recall.

**Exercise 2 — Regularisation sweep**  
`LogisticRegression` in sklearn takes a `C` parameter (inverse regularisation strength). Train models with `C ∈ [0.001, 0.01, 0.1, 1, 10, 100]`. Plot train vs. test AUC-ROC. At what C does the model start overfitting?

**Exercise 3 — Multi-class extension**  
Create a 3-class target: `low` (bottom 33%), `mid`, `high` (top 33%). Fit a `LogisticRegression(multi_class='multinomial')` and print the confusion matrix. Which class is hardest to distinguish?

In [ ]:
# ── Exercise 1 scaffold — find τ for precision ≥ 0.90 ─────────────────────
target_precision = 0.90

# YOUR CODE: iterate over thresholds_ex1 and find the lowest τ
# where precision_score(y_test, preds) >= target_precision
thresholds_ex1 = np.linspace(0.05, 0.99, 200)

# ...
# Expected: τ somewhere above 0.6, recall will drop noticeably

In [ ]:
# ── Exercise 2 scaffold — regularisation sweep ────────────────────────────
from sklearn.metrics import roc_auc_score

C_values = [0.001, 0.01, 0.1, 1, 10, 100]
train_aucs, test_aucs = [], []

for C in C_values:
    # YOUR CODE: fit LogisticRegression(C=C, max_iter=1000),
    # compute train and test AUC-ROC, append to lists
    pass

# YOUR CODE: plot train vs test AUC on a log-scale x-axis
# ...

In [ ]:
# ── Exercise 3 scaffold — 3-class target ──────────────────────────────────
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import ConfusionMatrixDisplay

# Build 3-class target using quantile cuts
y3 = pd.qcut(df["MedHouseVal"], q=3, labels=[0, 1, 2]).astype(int).values

# YOUR CODE: split, scale, fit LR(multi_class='multinomial', max_iter=1000)
# plot confusion matrix, identify which class is most confused
# ...